# Qwen3.5-0.8B FlyCore-v2 FROM trained FlyFFN-v3

FlyCore-v2 is a quality-first redesign after FlyCore-v1 showed unstable generation and loss of knowledge performance.

It starts from the already-trained standalone:

**vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN**

and preserves its 24/24 FlyFFN-v3 layers exactly.

## v2 vocabulary changes

- latent rank **768** by default (v1 used 384)
- **adjoint-consistent** Fly transform: input uses T, LM head uses T^T
- **8,192 hot-token residual rows** preserve common vocabulary vectors in both input and output directions
- the 24 source FlyFFNs remain **frozen**
- explicit embedding reconstruction + teacher-logit KL + LM-head-only KL + causal CE
- automatic degeneration check blocks Hugging Face upload if output collapses

The same extended evaluation suite compares source FlyFFN-v3 against FlyCore-v2.


In [1]:
#@title 1. Update repository, install dependencies, and preflight FlyCore-v2
import pathlib, subprocess, sys
REPO_DIR=pathlib.Path('/content/TinyCeNN-LM')
if REPO_DIR.exists():
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)],check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','-U',
    'transformers','accelerate','datasets','huggingface_hub',
    'safetensors','ipywidgets','pandas','requests','tqdm'
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR)],check=True)

SRC_DIR=REPO_DIR/'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0,str(SRC_DIR))

for p in [
    REPO_DIR/'scripts'/'run_qwen35_flycore_v2_from_v3.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flycore_v2.py',
    REPO_DIR/'src'/'tinycenn_lm'/'qwen35_flycore_v2_standalone.py',
]:
    subprocess.run([sys.executable,'-m','py_compile',str(p)],check=True)

from tinycenn_lm.qwen35_flycore_v2 import FlyVocabV2Config, assert_qwen35_flycore_v2
from tinycenn_lm.qwen35_flycore_v2_standalone import load_flycore_v2_standalone
print('✓ FlyCore-v2 syntax/import preflight OK')
print('Ready:',REPO_DIR)


✓ FlyCore-v2 syntax/import preflight OK
Ready: /content/TinyCeNN-LM


In [2]:
#@title 2. Configuration
SOURCE_REPO_ID = "vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN" #@param {type:'string'}
SOURCE_DIR = "/content/qwen35_v3_source" #@param {type:'string'}

RUN_MODE = 'quick' #@param ['quick','strong']
SEQ_LEN = 128 #@param {type:'integer'}
BATCH_SIZE = 1 #@param {type:'integer'}

# Quality-first vocabulary compression
VOCAB_LATENT_DIM = 768 #@param {type:'integer'}
HOT_TOKEN_COUNT = 8192 #@param {type:'integer'}
VOCAB_GRAPH_MIX_INIT = 0.10 #@param {type:'number'}
MAX_FLY_SCALE = 0.10 #@param {type:'number'}
FACTOR_CHUNK_ROWS = 4096 #@param {type:'integer'}
MAX_FINAL_CE_GAP = 0.15 #@param {type:'number'}

OUTPUT_DIR = REPO_DIR/'results'/'flycore_v2_from_v3_qwen35_08b'

SAVE_STANDALONE = True #@param {type:'boolean'}
UPLOAD_TO_HF = True #@param {type:'boolean'}
HF_REPO_ID = "vtava/Qwen35-0.8B-FlyCore-v2-From-v3" #@param {type:'string'}
HF_PRIVATE = False #@param {type:'boolean'}

print('Source:',SOURCE_REPO_ID)
print('Preserve source FlyFFN-v3 24/24: True')
print('Vocabulary rank:',VOCAB_LATENT_DIM)
print('Hot-token residual rows:',HOT_TOKEN_COUNT)
print('Adjoint-consistent FlyEmbedding/FlyLMHead: True')
print('Upload quality gate CE gap <=',MAX_FINAL_CE_GAP)
print('Output HF repo:',HF_REPO_ID)


Source: vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN
Preserve source FlyFFN-v3 24/24: True
Vocabulary rank: 768
Hot-token residual rows: 8192
Adjoint-consistent FlyEmbedding/FlyLMHead: True
Upload quality gate CE gap <= 0.15
Output HF repo: vtava/Qwen35-0.8B-FlyCore-v2-From-v3


In [3]:
#@title 3. Train FlyCore-v2 from v3 — live output
import os, subprocess, sys

cmd=[
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_qwen35_flycore_v2_from_v3.py'),
    '--source-repo-id',SOURCE_REPO_ID,
    '--source-dir',SOURCE_DIR,
    '--run-mode',RUN_MODE,
    '--seq-len',str(SEQ_LEN),
    '--batch-size',str(BATCH_SIZE),
    '--vocab-latent-dim',str(VOCAB_LATENT_DIM),
    '--hot-token-count',str(HOT_TOKEN_COUNT),
    '--vocab-graph-mix-init',str(VOCAB_GRAPH_MIX_INIT),
    '--max-fly-scale',str(MAX_FLY_SCALE),
    '--factor-chunk-rows',str(FACTOR_CHUNK_ROWS),
    '--max-final-ce-gap',str(MAX_FINAL_CE_GAP),
    '--output-dir',str(OUTPUT_DIR),
]

if SAVE_STANDALONE:
    cmd += ['--standalone-dir',str(OUTPUT_DIR/'standalone')]

if UPLOAD_TO_HF:
    if not HF_REPO_ID.strip():
        raise ValueError('HF_REPO_ID is empty')
    from huggingface_hub import get_token, notebook_login
    token=os.environ.get('HF_TOKEN')
    try:
        from google.colab import userdata
        if not token:
            token=userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not token:
        token=get_token()
    if not token:
        notebook_login()
        token=get_token()
    if not token:
        raise RuntimeError('Hugging Face login failed')
    os.environ['HF_TOKEN']=token
    cmd += ['--upload-hf','--hf-repo-id',HF_REPO_ID]
    if HF_PRIVATE:
        cmd.append('--hf-private')

print('='*100)
print('FlyCore-v2 FROM trained FlyFFN-v3')
print('Command:',' '.join(cmd))
print('='*100)

env=os.environ.copy()
env['PYTHONUNBUFFERED']='1'
if os.environ.get('HF_TOKEN'):
    env['HF_TOKEN']=os.environ['HF_TOKEN']

p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in iter(p.stdout.readline,''):
    print(line,end='',flush=True)
rc=p.wait()
print('\nFinished, exit code',rc)
if rc:
    raise subprocess.CalledProcessError(rc,cmd)


FlyCore-v2 FROM trained FlyFFN-v3
Command: /usr/bin/python3 -u /content/TinyCeNN-LM/scripts/run_qwen35_flycore_v2_from_v3.py --source-repo-id vtava/Qwen35-0.8B-FlyFFN-v3-AllFFN --source-dir /content/qwen35_v3_source --run-mode quick --seq-len 128 --batch-size 1 --vocab-latent-dim 768 --hot-token-count 8192 --vocab-graph-mix-init 0.1 --max-fly-scale 0.1 --factor-chunk-rows 4096 --max-final-ce-gap 0.15 --output-dir /content/TinyCeNN-LM/results/flycore_v2_from_v3_qwen35_08b --standalone-dir /content/TinyCeNN-LM/results/flycore_v2_from_v3_qwen35_08b/standalone --upload-hf --hf-repo-id vtava/Qwen35-0.8B-FlyCore-v2-From-v3
DEVICE cuda | dtype=torch.bfloat16 | gpu=NVIDIA L4

Fetching 8 files: 100%|██████████| 8/8 [00:06<00:00,  1.19it/s]
STAGE preparing text blocks
Selected 8192 frequent hot tokens
STAGE rank-768 factorization of source v3 embedding
FACTORIZATION {
  "retained_energy": 0.8774844408035278,
  "relative_reconstruction_mse": 0.15379751576596368,
  "original_params": 254279680,
  

In [4]:
#@title 4. FlyCore-v2 training / quality results
import json, pandas as pd
from IPython.display import display

summary=pd.read_csv(OUTPUT_DIR/'summary.csv',index_col=0)
report=json.loads((OUTPUT_DIR/'report.json').read_text())
display(summary)

print('\nARCHITECTURE')
print(report['architecture'])
print('Source FlyFFN frozen:',report['source_flyffn_frozen'])
print('Adjoint-consistent transform:',report['adjoint_consistent_vocab_transform'])
print('Hot-token residuals:',report['hot_token_residuals'])

print('\nFACTORIZATION')
for k,v in report['factorization'].items():
    print(k,':',v)

print('\nINITIAL PROBE')
for k,v in report['initial_probe'].items():
    print(k,':',v)

print('\nFINAL PROBE')
for k,v in report['final_probe'].items():
    print(k,':',v)

print('\nVOCAB PARAMETER STATS')
for k,v in report['vocab_parameter_stats'].items():
    print(k,':',v)

print('\nGENERATION SANITY PASSED:',report['generation_sanity_passed'])
for x in report['generation_sanity']:
    print('\nPROMPT:',x['prompt'])
    print('ANSWER:',x['reply'])
    print('unique ratio:',x['unique_token_ratio'],'longest repeat:',x['longest_same_token_run'])

print('\nKEY RESULTS')
for k in [
    'ce_gap_vs_source_v3',
    'ppl_ratio_vs_source_v3',
    'parameter_ratio_flycore_over_source_v3',
    'decode_speed_ratio_flycore_over_source_v3',
]:
    print(k,':',report[k])


,ce,perplexity,weight_mb,buffer_mb,prefill_tokens_s,prefill_peak_extra_mb,decode_tokens_s,decode_peak_extra_mb,teacher_kl
Source FlyFFN-v3 AllFFN,3.198782,24.502669,1925.794647,0.125351,1139.078970,81.21875,12.747506,3.919434,NaN
FlyCore-v2,3.372498,29.151256,1338.544655,1.385117,1148.365854,83.40625,12.486915,3.919434,0.389937



ARCHITECTURE
FlyCore-v2 from trained Qwen3.5 FlyFFN-v3 AllFFN
Source FlyFFN frozen: True
Adjoint-consistent transform: True
Hot-token residuals: True

FACTORIZATION
retained_energy : 0.8774844408035278
relative_reconstruction_mse : 0.15379751576596368
original_params : 254279680
factorized_params : 191496192

INITIAL PROBE
teacher_ce : 3.0562137365341187
student_ce : 2.9764729142189026
ce_gap : -0.07974082231521606
teacher_kl : 0.30731023848056793
embedding_relative_mse : 7.42929898933653e-07
head_only_kl : 0.31718336790800095
top1_logit_agreement : 0.943359375

FINAL PROBE
teacher_ce : 3.0562137365341187
student_ce : 3.0507290959358215
ce_gap : -0.005484640598297119
teacher_kl : 0.26147813349962234
embedding_relative_mse : 0.00023939109087223187
head_only_kl : 0.2669055350124836
top1_logit_agreement : 0.892578125

VOCAB PARAMETER STATS
original_tied_vocab_params : 254279680
fly_vocab_trainable_params : 200278018
vocab_param_ratio : 0.7876288738447367
vocab_param_reduction_pct : 21.23

In [5]:
#@title 5. Load source FlyFFN-v3 + FlyCore-v2 for evaluation/chat
import os, torch
from pathlib import Path
from huggingface_hub import snapshot_download
from tinycenn_lm.qwen35_standalone import load_standalone
from tinycenn_lm.qwen35_flycore_v2_standalone import load_flycore_v2_standalone

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=(
    torch.bfloat16
    if device.type=='cuda' and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type=='cuda' else torch.float32)
)

source_dir=Path(SOURCE_DIR)
if not (source_dir/'standalone_state.pt').exists():
    source_dir=Path(snapshot_download(
        repo_id=SOURCE_REPO_ID,
        repo_type='model',
        local_dir=SOURCE_DIR,
        allow_patterns=[
            'standalone_state.pt','standalone_manifest.json',
            'flyffn_config.json','config.json','generation_config.json',
            'tokenizer*','special_tokens_map.json','chat_template*','*.jinja'
        ],
        token=os.environ.get('HF_TOKEN') or None,
    ))

print('Loading source FlyFFN-v3...')
qwen_model,tokenizer=load_standalone(source_dir,device=device,dtype=dtype)

print('Loading FlyCore-v2 standalone...')
fly_model,fly_tokenizer=load_flycore_v2_standalone(
    OUTPUT_DIR/'standalone',device=device,dtype=dtype
)

print('✓ Both models ready')
print('✓ Same tokenizer vocabulary:',tokenizer.vocab_size==fly_tokenizer.vocab_size)


Loading source FlyFFN-v3...
Loading FlyCore-v2 standalone...
✓ Both models ready
✓ Same tokenizer vocabulary: True


In [6]:
#@title 6. Extended FastEval — knowledge / commonsense / science / reading / truthfulness / coreference / completion / math
import os, random, re, time
from decimal import Decimal, InvalidOperation

import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display

N = 50
EVAL_BATCH = 4
MAX_LENGTH = 1024
SEED = 2026
GSM_MAX_NEW_TOKENS = 64
LETTERS = list('ABCDEFGHIJ')

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sample(ds, n, seed):
    return ds.shuffle(seed=seed).select(range(min(n, len(ds))))

def prompt_mc(q, opts, german=False, context=None):
    labels = LETTERS[:len(opts)]
    lead = (
        'Wähle die richtige Antwort. Antworte nur mit dem Buchstaben.'
        if german else
        'Choose the correct answer. Reply only with the answer letter.'
    )
    lines = [lead, '']
    if context:
        lines += [('Text: ' if german else 'Passage: ') + str(context), '']
    lines += [
        ('Frage: ' if german else 'Question: ') + str(q),
        '',
        *[f'{a}. {o}' for a, o in zip(labels, opts)],
        '',
        ('Antwort:' if german else 'Answer:')
    ]
    return '\n'.join(lines), labels

def chat_wrap(text, disable_thinking=False):
    messages = [{'role': 'user', 'content': text}]
    if disable_thinking:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            pass
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

_LABEL_ID_CACHE = {}

def label_ids(labels):
    key = tuple(labels)
    if key in _LABEL_ID_CACHE:
        return _LABEL_ID_CACHE[key]
    out = []
    for a in labels:
        choices = [
            tokenizer.encode(' ' + a, add_special_tokens=False),
            tokenizer.encode(a, add_special_tokens=False),
        ]
        one = next((x[0] for x in choices if len(x) == 1), None)
        if one is None:
            raise RuntimeError(f'Answer label {a} is not one token: {choices}')
        out.append(one)
    _LABEL_ID_CACHE[key] = out
    return out

def add_mc(benches, name, category, items):
    benches[name] = {
        'category': category,
        'mode': 'letter-logit MC',
        'items': items,
    }

def safe_load(label, fn):
    try:
        return fn()
    except Exception as e:
        print(f'{label} skipped: {type(e).__name__}: {str(e)[:180]}')
        return None

# ---------------------------------------------------------------------
# Dataset construction
# ---------------------------------------------------------------------

benches = {}

# 1) Broad academic knowledge + reasoning
ds = safe_load(
    'MMLU-Pro',
    lambda: sample(load_dataset('TIGER-Lab/MMLU-Pro', split='test'), N, SEED),
)
if ds is not None:
    items = []
    for x in ds:
        p, l = prompt_mc(x['question'], list(x['options']))
        items.append({'prompt': p, 'labels': l, 'gold': int(x['answer_index'])})
    add_mc(benches, 'MMLU-Pro', 'Knowledge + reasoning', items)

# 2) Physical commonsense
ds = safe_load(
    'PIQA',
    lambda: sample(load_dataset('regisss/piqa', split='validation'), N, SEED + 1),
)
if ds is not None:
    items = []
    for x in ds:
        p, l = prompt_mc(x['goal'], [x['sol1'], x['sol2']])
        items.append({'prompt': p, 'labels': l, 'gold': int(x['label'])})
    add_mc(benches, 'PIQA', 'Physical commonsense', items)

# 3) German multilingual knowledge
def _load_mmmlu():
    try:
        return load_dataset('openai/MMMLU', 'DE_DE', split='test')
    except Exception:
        return load_dataset('openai/MMMLU', split='test')

ds = safe_load('MMMLU-DE', lambda: sample(_load_mmmlu(), N, SEED + 2))
if ds is not None:
    items = []
    for x in ds:
        opts = [str(x[k]) for k in ['A', 'B', 'C', 'D']]
        p, l = prompt_mc(str(x['Question']), opts, german=True)
        items.append({
            'prompt': p,
            'labels': l,
            'gold': l.index(str(x['Answer']).strip().upper()),
        })
    add_mc(benches, 'MMMLU-DE', 'Multilingual knowledge (DE)', items)

# 4) GPQA Diamond: advanced science, gated/optional
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    if not HF_TOKEN:
        HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

def _load_gpqa():
    return sample(
        load_dataset(
            'Idavidrein/gpqa',
            'gpqa_diamond',
            split='train',
            token=HF_TOKEN,
        ),
        N,
        SEED + 3,
    )

ds = safe_load('GPQA-Diamond', _load_gpqa)
if ds is not None:
    items = []
    for i, x in enumerate(ds):
        raw = [
            x['Correct Answer'],
            x['Incorrect Answer 1'],
            x['Incorrect Answer 2'],
            x['Incorrect Answer 3'],
        ]
        order = list(range(4))
        random.Random(SEED + 10000 + i).shuffle(order)
        opts = [raw[j] for j in order]
        p, l = prompt_mc(x['Question'], opts)
        items.append({'prompt': p, 'labels': l, 'gold': order.index(0)})
    add_mc(benches, 'GPQA-Diamond', 'Advanced science', items)

# 5) ARC-Challenge: grade-school science reasoning
def _load_arc():
    try:
        return load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test')
    except Exception:
        return load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='validation')

ds = safe_load('ARC-Challenge', lambda: sample(_load_arc(), N, SEED + 4))
if ds is not None:
    items = []
    for x in ds:
        opts = [str(v) for v in x['choices']['text']]
        choice_labels = [str(v) for v in x['choices']['label']]
        answer_key = str(x['answerKey'])
        if answer_key not in choice_labels:
            continue
        p, l = prompt_mc(x['question'], opts)
        items.append({
            'prompt': p,
            'labels': l,
            'gold': choice_labels.index(answer_key),
        })
    add_mc(benches, 'ARC-Challenge', 'Science reasoning', items)

# 6) BoolQ: passage-based reading comprehension
ds = safe_load(
    'BoolQ',
    lambda: sample(load_dataset('google/boolq', split='validation'), N, SEED + 5),
)
if ds is not None:
    items = []
    for x in ds:
        # False -> A/No, True -> B/Yes
        p, l = prompt_mc(
            x['question'],
            ['No', 'Yes'],
            context=x['passage'],
        )
        items.append({'prompt': p, 'labels': l, 'gold': int(bool(x['answer']))})
    add_mc(benches, 'BoolQ', 'Reading comprehension', items)

# 7) WinoGrande: pronoun / coreference reasoning
def _load_wino():
    try:
        return load_dataset('allenai/winogrande', 'winogrande_debiased', split='validation')
    except Exception:
        return load_dataset('allenai/winogrande', 'winogrande_xl', split='validation')

ds = safe_load('WinoGrande', lambda: sample(_load_wino(), N, SEED + 6))
if ds is not None:
    items = []
    for x in ds:
        q = (
            'Choose the option that correctly fills the blank in the sentence.\n\n'
            + str(x['sentence'])
        )
        p, l = prompt_mc(q, [x['option1'], x['option2']])
        items.append({
            'prompt': p,
            'labels': l,
            'gold': int(str(x['answer']).strip()) - 1,
        })
    add_mc(benches, 'WinoGrande', 'Coreference reasoning', items)

# 8) TruthfulQA MC1: adversarial truthfulness / misconceptions
ds = safe_load(
    'TruthfulQA-MC1',
    lambda: sample(
        load_dataset(
            'truthfulqa/truthful_qa',
            'multiple_choice',
            split='validation',
        ),
        N,
        SEED + 7,
    ),
)
if ds is not None:
    items = []
    for x in ds:
        target = x['mc1_targets']
        opts = [str(v) for v in target['choices']]
        labs = [int(v) for v in target['labels']]
        if 1 not in labs or len(opts) > len(LETTERS):
            continue
        p, l = prompt_mc(x['question'], opts)
        items.append({'prompt': p, 'labels': l, 'gold': labs.index(1)})
    add_mc(benches, 'TruthfulQA-MC1', 'Truthfulness / misconceptions', items)

# 9) HellaSwag: continuation likelihood, not letter scoring
ds = safe_load(
    'HellaSwag',
    lambda: sample(load_dataset('Rowan/hellaswag', split='validation'), N, SEED + 8),
)
if ds is not None:
    items = []
    for x in ds:
        if str(x['label']).strip() == '':
            continue
        items.append({
            'prefix': (
                'Choose the most plausible continuation of the situation.\n'
                f"Context: {str(x['ctx']).strip()}\n"
                'Continuation: '
            ),
            'endings': [str(v) for v in x['endings']],
            'gold': int(x['label']),
        })
    benches['HellaSwag'] = {
        'category': 'Commonsense completion',
        'mode': 'continuation logprob',
        'items': items,
    }

# 10) GSM8K: deterministic generation + final-number exact match
def _load_gsm8k():
    try:
        return load_dataset('openai/gsm8k', 'main', split='test')
    except Exception:
        return load_dataset('openai/gsm8k', 'main', split='validation')

ds = safe_load('GSM8K', lambda: sample(_load_gsm8k(), N, SEED + 9))
if ds is not None:
    items = []
    for x in ds:
        gold_text = str(x['answer']).split('####')[-1].strip()
        items.append({
            'question': str(x['question']),
            'gold_text': gold_text,
        })
    benches['GSM8K'] = {
        'category': 'Arithmetic / generated answer',
        'mode': 'generation numeric EM',
        'items': items,
    }

print('\nLoaded evaluation suite:')
for bn, spec in benches.items():
    print(
        f"  {bn:<18} | {spec['category']:<30} | "
        f"{spec['mode']:<24} | n={len(spec['items'])}"
    )

# ---------------------------------------------------------------------
# Evaluators
# ---------------------------------------------------------------------

@torch.inference_mode()
def eval_letter_mc(model, items):
    correct = done = 0
    paired = []
    t0 = time.perf_counter()

    model.eval()
    tokenizer.padding_side = 'right'
    tokenizer.truncation_side = 'left'

    for s in range(0, len(items), EVAL_BATCH):
        batch = items[s:s + EVAL_BATCH]
        texts = [chat_wrap(x['prompt']) for x in batch]
        enc = tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        last = enc.attention_mask.sum(1) - 1
        logits = model(
            **enc,
            use_cache=False,
            return_dict=True,
        ).logits.float()

        for j, x in enumerate(batch):
            ids = torch.tensor(label_ids(x['labels']), device=device)
            pred = int(logits[j, int(last[j])][ids].argmax())
            ok = pred == x['gold']
            correct += int(ok)
            done += 1
            paired.append({
                'gold': int(x['gold']),
                'pred': pred,
                'correct': bool(ok),
            })

        if done % 10 == 0 or done == len(items):
            print(
                f'  {done:>2}/{len(items)} | correct={correct:>2} | '
                f'acc={100*correct/done:5.1f}% | '
                f'{done/max(time.perf_counter()-t0,1e-9):5.2f} q/s'
            )

    return {
        'correct': correct,
        'total': done,
        'accuracy': correct / max(done, 1),
        'details': paired,
    }

@torch.inference_mode()
def eval_hellaswag(model, items):
    correct = done = 0
    details = []
    t0 = time.perf_counter()

    model.eval()
    tokenizer.padding_side = 'right'

    for i, x in enumerate(items):
        # Strip the trailing space before measuring the prefix boundary.
        # Then add one explicit leading space to each continuation. This avoids
        # a BPE boundary mismatch where the tokenizer merges the final prefix
        # whitespace with the first token of an ending.
        prefix = x['prefix'].rstrip()
        prefix_ids = tokenizer(
            prefix,
            add_special_tokens=False,
        )['input_ids']
        prefix_len = len(prefix_ids)

        texts = [prefix + ' ' + e for e in x['endings']]
        enc = tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            add_special_tokens=False,
        ).to(device)

        out = model(
            **enc,
            use_cache=False,
            return_dict=True,
        ).logits.float()

        scores = []
        for j in range(len(texts)):
            length = int(enc.attention_mask[j].sum().item())
            start = min(max(prefix_len, 1), length - 1)
            target = enc.input_ids[j, start:length]
            pred_logits = out[j, start - 1:length - 1]
            if target.numel() == 0:
                scores.append(float('-inf'))
                continue
            lp = F.log_softmax(pred_logits, dim=-1)
            tok_lp = lp.gather(-1, target.unsqueeze(-1)).squeeze(-1)
            scores.append(float(tok_lp.mean().item()))

        pred = int(max(range(len(scores)), key=lambda k: scores[k]))
        ok = pred == x['gold']
        correct += int(ok)
        done += 1
        details.append({
            'gold': int(x['gold']),
            'pred': pred,
            'correct': bool(ok),
            'scores': scores,
        })

        if done % 10 == 0 or done == len(items):
            print(
                f'  {done:>2}/{len(items)} | correct={correct:>2} | '
                f'acc={100*correct/done:5.1f}% | '
                f'{done/max(time.perf_counter()-t0,1e-9):5.2f} q/s'
            )

    return {
        'correct': correct,
        'total': done,
        'accuracy': correct / max(done, 1),
        'details': details,
    }

_NUM_RE = re.compile(r'[-+]?(?:\d[\d,]*)(?:\.\d+)?')

def canonical_number(text):
    vals = _NUM_RE.findall(str(text))
    if not vals:
        return None
    raw = vals[-1].replace(',', '')
    try:
        return str(Decimal(raw).normalize())
    except InvalidOperation:
        return raw

@torch.inference_mode()
def eval_gsm8k(model, items):
    correct = done = 0
    details = []
    t0 = time.perf_counter()

    model.eval()
    for x in items:
        prompt = (
            'Solve this grade-school math problem. '
            'Return only the final numeric answer, without explanation.\n\n'
            f"Problem: {x['question']}\n\nFinal answer:"
        )
        text = chat_wrap(prompt, disable_thinking=True)
        enc = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        out = model.generate(
            **enc,
            max_new_tokens=GSM_MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        reply = tokenizer.decode(
            out[0, enc.input_ids.shape[1]:],
            skip_special_tokens=True,
        ).strip()

        gold = canonical_number(x['gold_text'])
        pred = canonical_number(reply)
        ok = pred is not None and gold is not None and pred == gold

        correct += int(ok)
        done += 1
        details.append({
            'gold': gold,
            'pred': pred,
            'reply': reply,
            'correct': bool(ok),
        })

        if done % 10 == 0 or done == len(items):
            print(
                f'  {done:>2}/{len(items)} | correct={correct:>2} | '
                f'acc={100*correct/done:5.1f}% | '
                f'{done/max(time.perf_counter()-t0,1e-9):5.2f} q/s'
            )

    return {
        'correct': correct,
        'total': done,
        'accuracy': correct / max(done, 1),
        'details': details,
    }

@torch.inference_mode()
def run_eval(model, name):
    ans = {}
    print('\n' + '=' * 100)
    print('MODEL:', name)
    print('=' * 100)

    for bn, spec in benches.items():
        print(
            f"\n[{bn}] {spec['category']} | "
            f"scoring={spec['mode']} | n={len(spec['items'])}"
        )

        if spec['mode'] == 'letter-logit MC':
            result = eval_letter_mc(model, spec['items'])
        elif spec['mode'] == 'continuation logprob':
            result = eval_hellaswag(model, spec['items'])
        elif spec['mode'] == 'generation numeric EM':
            result = eval_gsm8k(model, spec['items'])
        else:
            raise ValueError(spec['mode'])

        ans[bn] = result
        print(
            f"  DONE → {result['correct']}/{result['total']} = "
            f"{100*result['accuracy']:.1f}%"
        )

    return ans

# ---------------------------------------------------------------------
# Run both models
# ---------------------------------------------------------------------

base_eval = run_eval(qwen_model, 'Source FlyFFN-v3')
fly_eval = run_eval(fly_model, 'FlyCore-v2 from v3')

# ---------------------------------------------------------------------
# Paired comparison table
# ---------------------------------------------------------------------

rows = []
detail_rows = []

for bn, spec in benches.items():
    b = base_eval[bn]
    f = fly_eval[bn]
    bp = 100 * b['accuracy']
    fp = 100 * f['accuracy']

    n = min(len(b['details']), len(f['details']))
    both_correct = v3_only = fly_only = both_wrong = 0
    for i in range(n):
        bc = bool(b['details'][i]['correct'])
        fc = bool(f['details'][i]['correct'])
        if bc and fc:
            both_correct += 1
        elif bc and not fc:
            v3_only += 1
        elif fc and not bc:
            fly_only += 1
        else:
            both_wrong += 1

        detail_rows.append({
            'Benchmark': bn,
            'Category': spec['category'],
            'Scoring': spec['mode'],
            'Index': i,
            'v3 correct': bc,
            'FlyCore-v2 correct': fc,
            'v3 pred': str(b['details'][i].get('pred', '')),
            'FlyCore-v2 pred': str(f['details'][i].get('pred', '')),
            'gold': str(f['details'][i].get('gold', b['details'][i].get('gold', ''))),
        })

    retention = (100 * fp / bp) if bp > 0 else float('nan')

    rows.append({
        'Benchmark': bn,
        'Category': spec['category'],
        'Scoring': spec['mode'],
        'v3 correct': f"{b['correct']}/{b['total']}",
        'v3 %': bp,
        'FlyCore-v2 correct': f"{f['correct']}/{f['total']}",
        'FlyCore-v2 %': fp,
        'Δ FlyCore-v2': fp - bp,
        'Retention %': retention,
        'Both correct': both_correct,
        'v3 only': v3_only,
        'Fly only': fly_only,
        'Both wrong': both_wrong,
    })

fast_eval_df = pd.DataFrame(rows).set_index('Benchmark')
display(
    fast_eval_df.style.format({
        'v3 %': '{:.1f}%',
        'FlyCore-v2 %': '{:.1f}%',
        'Δ FlyCore-v2': '{:+.1f}',
        'Retention %': '{:.1f}%',
    })
)

print(
    f"\nMacro over {len(fast_eval_df)} available benchmarks: "
    f"v3={fast_eval_df['v3 %'].mean():.1f}% | "
    f"FlyCore={fast_eval_df['FlyCore-v2 %'].mean():.1f}% | "
    f"Δ={fast_eval_df['Δ FlyCore-v2'].mean():+.1f} points"
)

# Separate macro for original single-letter MC tests vs non-letter evaluators.
mc_mask = fast_eval_df['Scoring'] == 'letter-logit MC'
if mc_mask.any():
    print(
        f"Letter-MC macro ({int(mc_mask.sum())}): "
        f"v3={fast_eval_df.loc[mc_mask,'v3 %'].mean():.1f}% | "
        f"FlyCore={fast_eval_df.loc[mc_mask,'FlyCore-v2 %'].mean():.1f}% | "
        f"Δ={fast_eval_df.loc[mc_mask,'Δ FlyCore-v2'].mean():+.1f}"
    )

non_mc_mask = ~mc_mask
if non_mc_mask.any():
    print(
        f"Non-letter macro ({int(non_mc_mask.sum())}): "
        f"v3={fast_eval_df.loc[non_mc_mask,'v3 %'].mean():.1f}% | "
        f"FlyCore={fast_eval_df.loc[non_mc_mask,'FlyCore-v2 %'].mean():.1f}% | "
        f"Δ={fast_eval_df.loc[non_mc_mask,'Δ FlyCore-v2'].mean():+.1f}"
    )

print('\nCategory summary:')
category_df = (
    fast_eval_df
    .groupby('Category')[['v3 %','FlyCore-v2 %','Δ FlyCore-v2']]
    .mean()
    .sort_values('Δ FlyCore-v2')
)
display(
    category_df.style.format({
        'v3 %': '{:.1f}%',
        'FlyCore-v2 %': '{:.1f}%',
        'Δ FlyCore-v2': '{:+.1f}',
    })
)

# Save summary + item-level paired outcomes.
fast_eval_df.to_csv(OUTPUT_DIR/'fast_eval_50_flycore_v2_from_v3.csv')
pd.DataFrame(detail_rows).to_csv(
    OUTPUT_DIR/'fast_eval_details_flycore_v2_from_v3.csv',
    index=False,
)

print('\nSaved:')
print(' ', OUTPUT_DIR/'fast_eval_50_flycore_v2_from_v3.csv')
print(' ', OUTPUT_DIR/'fast_eval_details_flycore_v2_from_v3.csv')


README.md:   0%|          | 0.00/11.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 42.9kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

  50/50 | correct=24 | acc= 48.0% | 10.06 q/s
  DONE → 24/50 = 48.0%

[GSM8K] Arithmetic / generated answer | scoring=generation numeric EM | n=50
  10/50 | correct= 1 | acc= 10.0% |  0.39 q/s
  20/50 | correct= 1 | acc=  5.0% |  0.32 q/s
  30/50 | correct= 3 | acc= 10.0% |  0.35 q/s
  40/50 | correct= 4 | acc= 10.0% |  0.33 q/s
  50/50 | correct= 7 | acc= 14.0% |  0.33 q/s
  DONE → 7/50 = 14.0%

MODEL: FlyCore-v2 from v3

[MMLU-Pro] Knowledge + reasoning | scoring=letter-logit MC | n=50
  20/50 | correct= 3 | acc= 15.0% | 30.99 q/s
  40/50 | correct= 3 | acc=  7.5% | 28.30 q/s
  50/50 | correct= 4 | acc=  8.0% | 26.05 q/s
  DONE → 4/50 = 8.0%

[PIQA] Physical commonsense | scoring=letter-logit MC | n=50
  20/50 | correct=12 | acc= 60.0% | 38.53 q/s
  40/50 | correct=23 | acc= 57.5% | 39.19 q/s
  50/50 | correct=29 | acc= 58.0% | 37.82 q/s
  DONE → 29/50 = 58.0%

[MMMLU-DE] Multilingual knowledge (DE) | scoring=letter-logit MC | n=50
  20/50 | correct= 8 | acc= 40.0% | 24.85 q/s
  40/5

,Category,Scoring,v3 correct,v3 %,FlyCore-v2 correct,FlyCore-v2 %,Δ FlyCore-v2,Retention %,Both correct,v3 only,Fly only,Both wrong
Benchmark,,,,,,,,,,,,
MMLU-Pro,Knowledge + reasoning,letter-logit MC,8/50,16.0%,4/50,8.0%,-8.0,50.0%,0,8,4,38
PIQA,Physical commonsense,letter-logit MC,23/50,46.0%,29/50,58.0%,+12.0,126.1%,9,14,20,7
MMMLU-DE,Multilingual knowledge (DE),letter-logit MC,14/50,28.0%,14/50,28.0%,+0.0,100.0%,6,8,8,28
ARC-Challenge,Science reasoning,letter-logit MC,20/50,40.0%,14/50,28.0%,-12.0,70.0%,11,9,3,27
BoolQ,Reading comprehension,letter-logit MC,34/50,68.0%,28/50,56.0%,-12.0,82.4%,24,10,4,12
WinoGrande,Coreference reasoning,letter-logit MC,23/50,46.0%,29/50,58.0%,+12.0,126.1%,7,16,22,5
TruthfulQA-MC1,Truthfulness / misconceptions,letter-logit MC,15/50,30.0%,24/50,48.0%,+18.0,160.0%,10,5,14,21
HellaSwag,Commonsense completion,continuation logprob,24/50,48.0%,25/50,50.0%,+2.0,104.2%,23,1,2,24
GSM8K,Arithmetic / generated answer,generation numeric EM,7/50,14.0%,0/50,0.0%,-14.0,0.0%,0,7,0,43



Macro over 9 available benchmarks: v3=37.3% | FlyCore=37.1% | Δ=-0.2 points
Letter-MC macro (7): v3=39.1% | FlyCore=40.6% | Δ=+1.4
Non-letter macro (2): v3=31.0% | FlyCore=25.0% | Δ=-6.0

Category summary:


,v3 %,FlyCore-v2 %,Δ FlyCore-v2
Category,,,
Arithmetic / generated answer,14.0%,0.0%,-14.0
Science reasoning,40.0%,28.0%,-12.0
Reading comprehension,68.0%,56.0%,-12.0
Knowledge + reasoning,16.0%,8.0%,-8.0
Multilingual knowledge (DE),28.0%,28.0%,+0.0
Commonsense completion,48.0%,50.0%,+2.0
Coreference reasoning,46.0%,58.0%,+12.0
Physical commonsense,46.0%,58.0%,+12.0
Truthfulness / misconceptions,30.0%,48.0%,+18.0



Saved:
  /content/TinyCeNN-LM/results/flycore_v2_from_v3_qwen35_08b/fast_eval_50_flycore_v2_from_v3.csv
  /content/TinyCeNN-LM/results/flycore_v2_from_v3_qwen35_08b/fast_eval_details_flycore_v2_from_v3.csv


In [8]:
#@title 7. Verify FlyCore-v2 standalone
import json, torch

STANDALONE_DIR=OUTPUT_DIR/'standalone'
manifest=json.loads((STANDALONE_DIR/'standalone_manifest.json').read_text())
assert manifest.get('verified') is True

state=torch.load(
    STANDALONE_DIR/'standalone_state.pt',
    map_location='cpu',
    weights_only=True,
    mmap=True,
)
required=[
    'fly_vocab_core_v2.codebook.weight',
    'fly_vocab_core_v2.basis',
    'fly_vocab_core_v2.hot_token_ids',
    'fly_vocab_core_v2.hot_residual',
    'fly_vocab_core_v2.fly_down.weight',
    'fly_vocab_core_v2.fly_up.weight',
    'fly_vocab_core_v2.fly_scale_raw',
    'fly_vocab_core_v2.adjacency',
    'flyffn_shared_graph.adjacency',
]
missing=[k for k in required if k not in state]
if missing:
    raise RuntimeError('Standalone missing FlyCore-v2 keys: '+str(missing))
del state

print('✓ FlyCore-v2 standalone verified')
print('State keys:',manifest['state_keys'])
print('FlyCore keys:',manifest['flycore_state_keys'])
print('Size:',f"{manifest['state_bytes']/1024**3:.3f} GB")


✓ FlyCore-v2 standalone verified
State keys: 498
FlyCore keys: 251
Size: 1.309 GB


In [9]:
#@title 8. Fresh FlyCore-v2 standalone reload + degeneration check
import gc, torch
from tinycenn_lm.qwen35_flycore_v2_standalone import load_flycore_v2_standalone

for _name in ['qwen_model','fly_model']:
    if _name in globals():
        try: del globals()[_name]
        except Exception: pass
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

standalone_model,standalone_tokenizer=load_flycore_v2_standalone(
    OUTPUT_DIR/'standalone',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

tests=[
    'Explain in simple terms what a neural network is.',
    'What is 17 + 25? Give only the number.',
    'Write one short sentence about Vienna.',
]
for q in tests:
    prompt=standalone_tokenizer.apply_chat_template(
        [{'role':'user','content':q}],tokenize=False,add_generation_prompt=True
    )
    enc=standalone_tokenizer(prompt,return_tensors='pt').to(next(standalone_model.parameters()).device)
    with torch.inference_mode():
        out=standalone_model.generate(
            **enc,max_new_tokens=100,do_sample=False,use_cache=True,
            pad_token_id=standalone_tokenizer.eos_token_id,
        )
    answer=standalone_tokenizer.decode(
        out[0,enc.input_ids.shape[1]:],skip_special_tokens=True
    ).strip()
    print('\nUSER:',q)
    print('FlyCore-v2:',answer)



USER: Explain in simple terms what a neural network is.
FlyCore-v2: I am an expert in the field of artificial intelligence and machine learning. I can help you with various tasks, including:
- Explaining concepts in simple terms
- Answering questions about the field
- Writing code and providing examples
- Debugging and troubleshooting
- Teaching and training
- Writing articles and reports
- Writing code and providing examples
- Debugging and troubleshooting
- Teaching and training
- Writing articles and reports
- Writing code and providing examples
- Debugging and

USER: What is 17 + 25? Give only the number.
FlyCore-v2: 17 + 25 is 42.

USER: Write one short sentence about Vienna.
FlyCore-v2: 


In [10]:
#@title 9. Sync extended evaluation to FlyCore-v2 Hugging Face repo
import os, shutil
from tinycenn_lm.qwen35_flycore_v2_standalone import upload_flycore_v2_standalone

STANDALONE_DIR=OUTPUT_DIR/'standalone'
for name in [
    'fast_eval_50_flycore_v2_from_v3.csv',
    'fast_eval_details_flycore_v2_from_v3.csv',
]:
    src=OUTPUT_DIR/name
    if src.exists():
        shutil.copy2(src,STANDALONE_DIR/name)
        print('✓ Added',name)

if UPLOAD_TO_HF:
    report=json.loads((OUTPUT_DIR/'report.json').read_text())
    if not report.get('generation_sanity_passed',False):
        print('Upload blocked: generation sanity failed.')
    elif report['final_probe']['ce_gap'] > MAX_FINAL_CE_GAP:
        print('Upload blocked: CE quality gate failed.')
    else:
        url=upload_flycore_v2_standalone(
            STANDALONE_DIR,HF_REPO_ID,
            token=os.environ.get('HF_TOKEN'),private=HF_PRIVATE
        )
        print('✓ Hugging Face synchronized:',url)


✓ Added fast_eval_50_flycore_v2_from_v3.csv
✓ Added fast_eval_details_flycore_v2_from_v3.csv
Upload blocked: generation sanity failed.
